In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import gc
import json

import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.utils import class_weight
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

from lightgbm import LGBMClassifier

from utils import reduce_mem_usage

In [3]:
class ClassificationNet(nn.Module):
    def __init__(self, in_feature):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(in_feature, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5)
        )

        self.layer2 = nn.Sequential(
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5)
        )

        self.layer3 = nn.Sequential(
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        return out

In [4]:
class TorchPredictor(BaseEstimator, ClassifierMixin):
    def __init__(self, model, device="cpu"):
        self.model = model
        self.device = device

    def fit(self, X=None, y=None):
        # Модель уже обучена — просто переводим в eval и возвращаем себя
        self.model.to(self.device).eval()
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float32)
        self.model.eval()
        with torch.no_grad():
            x_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
            logits = self.model(x_tensor).squeeze(1)
            probs = torch.sigmoid(logits).cpu().numpy()
        return np.vstack([1 - probs, probs]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

In [5]:
model = ClassificationNet(in_feature=386)

In [6]:
model.load_state_dict(torch.load('models/nn.pth', map_location='cpu'))

<All keys matched successfully>

In [7]:
torch_pred = TorchPredictor(model, device="cpu")

In [8]:
with open(r'prep_data/unique_dict.txt', 'r', encoding='utf-8') as f:
    dict_key = json.load(f)
columns_for_enc = [dict_key[col] for col in dict_key.keys()]

In [9]:
feature_names = [f'{key}_{value}' for key, values in dict_key.items() for value in values[1:]]

In [10]:
cat_cols = list(dict_key.keys())

In [11]:
class MemoryReduceTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.stats_ = None

    def fit(self, X, y):
        self.stats_ = reduce_mem_usage(X)
        return self

    def transform(self, X):
        X = self.stats_
        return X

In [12]:
class AggregationTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self, df_init: pd.DataFrame, feature_names: list, group_col: str='id', agg_func: str='sum'):
        self.df_init = df_init
        self.feature_names = feature_names
        self.group_col = group_col
        self.agg_func = agg_func

    def fit(self, X, y):

        return self

    def transform(self, X):
        df_temp = pd.DataFrame(X, columns=self.feature_names).astype('int8')
        df_temp = pd.concat([self.df_init, df_temp], axis=1)
        df_temp = df_temp.groupby(self.group_col, as_index=False).agg(self.agg_func)
        df_temp = df_temp.drop(columns=['id'])
        X = df_temp.copy()
        return X

In [13]:
X = pd.read_parquet(r'raw_data/train_data_0.pq')
y = pd.read_csv(r'raw_data/target.csv')

In [14]:
X.shape, y.shape

((1974724, 61), (3000000, 2))

In [15]:
df = X.merge(y, how='inner', on='id')
df = df[:200000]
df.shape

(200000, 62)

In [16]:
y_test = df['flag']
X_test = df.drop(['flag'], axis=1)
X_test.shape, y_test.shape

((200000, 61), (200000,))

In [17]:
X_id = X_test[['id']].copy()

In [18]:
min_id, max_id = X_test['id'].min(), X_test['id'].max()
min_id, max_id

(0, 25880)

In [19]:
y_test_prep = df[(df['id'] >= min_id) & (df['id'] <= max_id)][['id', 'flag']].drop_duplicates(ignore_index=True)

In [20]:
y_test = y_test_prep.drop(['id'], axis=1)

In [21]:
categorical_transformer = Pipeline(steps=[
    ('red_mem', MemoryReduceTransformer()),
    ('onehot', OneHotEncoder(categories=columns_for_enc, sparse_output=False, handle_unknown='ignore', drop='first')),
    ('agg_trans', AggregationTransformer(df_init=X_id, feature_names=feature_names)),
    #('scaler', StandardScaler()),
])

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, cat_cols),
    ]
)

In [23]:
clf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', torch_pred)
])

In [24]:
clf.fit(X_test, y_test)

Используемая память до преобразования 0.089 Gb


  0%|          | 0/60 [00:00<?, ?it/s]

Используемая память после преобразования: 0.011 Gb
Количество используемой памяти уменьшено на 87.5%


,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
X_test.shape

(200000, 61)

In [26]:
y_pred_proba = clf.predict_proba(X_test)[:, 1]

In [27]:
auc_test = roc_auc_score(y_test, y_pred_proba)
print(f'AUC на тесте: {auc_test:.4f}')

AUC на тесте: 0.7844
